# School Site Suitability Analysis - Chennai
## Multi-Criteria GIS Analysis for High School Location Selection

**Study Area**: Chennai, Tamil Nadu, India
**Methodology**: Raster-based weighted overlay analysis
**Resolution**: 30 meters

---

## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.plot import show
from rasterio.mask import mask
from scipy import ndimage
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import seaborn as sns
import folium
from pathlib import Path

# Configuration
from config import *

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)

print('✓ Libraries imported successfully')

## 2. Data Generation

In [ ]:
# Check if data exists, if not generate it
if not (PROCESSED_DATA_DIR / OUTPUT_FILES['lulc_raster']).exists():
    print('Generating spatial datasets...')
    from scripts.data_generation import generate_all_data
    generate_all_data()
else:
    print('✓ Spatial datasets already exist')

# List generated files
print('\nGenerated files:')
for file in PROCESSED_DATA_DIR.glob('*'):
    print(f'  • {file.name}')

## 3. Load and Explore Spatial Data

In [ ]:
# Load raster datasets
with rasterio.open(PROCESSED_DATA_DIR / OUTPUT_FILES['lulc_raster']) as src:
    lulc = src.read(1)
    lulc_profile = src.profile
    transform = src.transform

with rasterio.open(PROCESSED_DATA_DIR / OUTPUT_FILES['population_raster']) as src:
    population = src.read(1)

with rasterio.open(PROCESSED_DATA_DIR / 'dem.tif') as src:
    dem = src.read(1)

with rasterio.open(PROCESSED_DATA_DIR / OUTPUT_FILES['hazard_raster']) as src:
    hazard = src.read(1)

print('✓ Raster datasets loaded')
print(f'  • LULC shape: {lulc.shape}')
print(f'  • Population shape: {population.shape}')
print(f'  • DEM shape: {dem.shape}')
print(f'  • Hazard shape: {hazard.shape}')

In [ ]:
# Load vector datasets
roads = gpd.read_file(RAW_DATA_DIR / 'road_network.shp')
schools = gpd.read_file(RAW_DATA_DIR / 'existing_schools.shp')
water = gpd.read_file(RAW_DATA_DIR / 'water_bodies.shp')

print('✓ Vector datasets loaded')
print(f'\nRoads: {len(roads)} features')
print(f'Schools: {len(schools)} features')
print(f'Water bodies: {len(water)} features')

print('\nSchools data:')
print(schools.head())

## 4. Visualize Input Data

In [ ]:
# Visualize LULC
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# LULC
im1 = axes[0, 0].imshow(lulc, cmap='tab10')
axes[0, 0].set_title('Land Use/Land Cover Classification', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Longitude')
axes[0, 0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0, 0], label='LULC Class')

# Population Density
im2 = axes[0, 1].imshow(population, cmap='YlOrRd')
axes[0, 1].set_title('Population Density', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Longitude')
axes[0, 1].set_ylabel('Latitude')
plt.colorbar(im2, ax=axes[0, 1], label='Persons/km²')

# DEM/Elevation
im3 = axes[1, 0].imshow(dem, cmap='terrain')
axes[1, 0].set_title('Digital Elevation Model', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
plt.colorbar(im3, ax=axes[1, 0], label='Elevation (m)')

# Hazard Zones
im4 = axes[1, 1].imshow(hazard, cmap='RdYlGn_r')
axes[1, 1].set_title('Hazard Zones (Flood-Prone Areas)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Longitude')
axes[1, 1].set_ylabel('Latitude')
plt.colorbar(im4, ax=axes[1, 1], label='Hazard Level')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'input_data_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Input data visualization saved')

## 5. Criteria Computation

In [ ]:
# 5.1 LULC Suitability
lulc_suitability = np.zeros_like(lulc, dtype=np.float32)
for i in range(1, 7):
    class_name = list(LULC_SUITABILITY.keys())[i-1] if i-1 < len(LULC_SUITABILITY) else 'other'
    score = list(LULC_SUITABILITY.values())[i-1] if i-1 < len(LULC_SUITABILITY) else 40
    lulc_suitability[lulc == i] = score

print('✓ LULC Suitability computed')
print(f'  Range: {lulc_suitability.min():.2f} - {lulc_suitability.max():.2f}')
print(f'  Mean: {lulc_suitability.mean():.2f}')

In [ ]:
# 5.2 Population Density Suitability
pop_normalized = np.clip(population / (population.max() + 1e-6), 0, 1)
pop_suitability = 100 * (1 - np.exp(-2 * pop_normalized))

print('✓ Population Suitability computed')
print(f'  Range: {pop_suitability.min():.2f} - {pop_suitability.max():.2f}')
print(f'  Mean: {pop_suitability.mean():.2f}')

In [ ]:
# 5.3 Slope Suitability from DEM
sx = ndimage.sobel(dem, axis=0)
sy = ndimage.sobel(dem, axis=1)
slope_degrees = np.degrees(np.arctan(np.sqrt(sx**2 + sy**2)))

slope_suitability = np.zeros_like(slope_degrees, dtype=np.float32)
slope_suitability[(slope_degrees >= 0) & (slope_degrees <= 2)] = 100
slope_suitability[(slope_degrees > 2) & (slope_degrees <= 5)] = 90
slope_suitability[(slope_degrees > 5) & (slope_degrees <= 15)] = 70
slope_suitability[(slope_degrees > 15) & (slope_degrees <= 30)] = 30
slope_suitability[slope_degrees > 30] = 0

print('✓ Slope Suitability computed')
print(f'  Slope range: {slope_degrees.min():.2f}° - {slope_degrees.max():.2f}°')
print(f'  Suitability range: {slope_suitability.min():.2f} - {slope_suitability.max():.2f}')
print(f'  Mean: {slope_suitability.mean():.2f}')

In [ ]:
# 5.4 Distance-based Suitability
# For accessibility: roads, schools, water

def create_distance_raster_from_shapefile(gdf, height, width, transform):
    """Create a distance raster from geometries"""
    from shapely.geometry import Point
    
    distance_raster = np.full((height, width), np.inf, dtype=np.float32)
    
    for geom in gdf.geometry:
        for i in range(0, height, 5):  # Sample every 5 pixels for speed
            for j in range(0, width, 5):
                point = Point(
                    transform.c + j * transform.a,
                    transform.f + i * transform.e
                )
                dist = point.distance(geom)
                distance_raster[i, j] = min(distance_raster[i, j], dist)
    
    return distance_raster

print('Computing distance rasters (this may take a moment)...')

# Road accessibility (closer = better)
road_distances = create_distance_raster_from_shapefile(roads, lulc.shape[0], lulc.shape[1], transform)
road_distances = np.where(np.isinf(road_distances), road_distances.max(), road_distances)
road_suitability = 100 * np.exp(-road_distances / 500)  # 500m characteristic length

# School buffer (farther = better, avoid redundancy)
school_distances = create_distance_raster_from_shapefile(schools, lulc.shape[0], lulc.shape[1], transform)
school_distances = np.where(np.isinf(school_distances), school_distances.max(), school_distances)
school_suitability = np.clip(100 * school_distances / (2 * BUFFERS['school_buffer']), 0, 100)

# Water buffer (farther = better)
water_distances = create_distance_raster_from_shapefile(water, lulc.shape[0], lulc.shape[1], transform)
water_distances = np.where(np.isinf(water_distances), water_distances.max(), water_distances)
water_suitability = np.clip(100 * water_distances / (2 * BUFFERS['water_buffer']), 0, 100)

print('✓ Distance-based suitability computed')

In [ ]:
# 5.5 Hazard Avoidance
hazard_suitability = (1 - hazard.astype(np.float32)) * 100

print('✓ Hazard Avoidance computed')
print(f'  Range: {hazard_suitability.min():.2f} - {hazard_suitability.max():.2f}')

## 6. Weighted Overlay - Combined Suitability Analysis

In [ ]:
# Combine all criteria using weighted overlay
combined_suitability = (
    CRITERIA_WEIGHTS['lulc_suitable'] * lulc_suitability +
    CRITERIA_WEIGHTS['population_density'] * pop_suitability +
    CRITERIA_WEIGHTS['distance_from_roads'] * road_suitability +
    CRITERIA_WEIGHTS['distance_from_schools'] * school_suitability +
    CRITERIA_WEIGHTS['distance_from_water'] * water_suitability +
    CRITERIA_WEIGHTS['slope_suitability'] * slope_suitability +
    CRITERIA_WEIGHTS['hazard_zones'] * hazard_suitability
)

# Normalize to 0-100
combined_suitability = np.clip(combined_suitability, 0, 100)

print('✓ Weighted Overlay Complete')
print(f'\nFinal Suitability Statistics:')
print(f'  Minimum: {combined_suitability.min():.2f}')
print(f'  Maximum: {combined_suitability.max():.2f}')
print(f'  Mean: {combined_suitability.mean():.2f}')
print(f'  Median: {np.median(combined_suitability):.2f}')
print(f'  Std Dev: {combined_suitability.std():.2f}')

## 7. Classification

In [ ]:
# Classify suitability into categories
classified_suitability = np.zeros_like(combined_suitability, dtype=np.uint8)

for class_idx, (class_name, (min_val, max_val, color)) in enumerate(SUITABILITY_CLASSES.items(), 1):
    mask = (combined_suitability >= min_val) & (combined_suitability < max_val)
    classified_suitability[mask] = class_idx
    count = np.sum(mask)
    percentage = (count / masked.size) * 100
    print(f'{class_name.title()}: {count:,} pixels ({percentage:.2f}%)')

print('✓ Classification complete')

## 8. Visualization of Results

In [ ]:
# Visualize all suitability layers
fig, axes = plt.subplots(3, 3, figsize=(18, 15))

# LULC Suitability
im = axes[0, 0].imshow(lulc_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[0, 0].set_title('LULC Suitability (20%)', fontweight='bold')
plt.colorbar(im, ax=axes[0, 0])

# Population Suitability
im = axes[0, 1].imshow(pop_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[0, 1].set_title('Population Density Suitability (15%)', fontweight='bold')
plt.colorbar(im, ax=axes[0, 1])

# Road Accessibility
im = axes[0, 2].imshow(road_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[0, 2].set_title('Road Accessibility (15%)', fontweight='bold')
plt.colorbar(im, ax=axes[0, 2])

# School Buffer
im = axes[1, 0].imshow(school_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[1, 0].set_title('School Redundancy Buffer (15%)', fontweight='bold')
plt.colorbar(im, ax=axes[1, 0])

# Water Buffer
im = axes[1, 1].imshow(water_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[1, 1].set_title('Water Body Buffer (15%)', fontweight='bold')
plt.colorbar(im, ax=axes[1, 1])

# Slope Suitability
im = axes[1, 2].imshow(slope_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[1, 2].set_title('Slope Suitability (12%)', fontweight='bold')
plt.colorbar(im, ax=axes[1, 2])

# Hazard Avoidance
im = axes[2, 0].imshow(hazard_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[2, 0].set_title('Hazard Avoidance (8%)', fontweight='bold')
plt.colorbar(im, ax=axes[2, 0])

# Combined Suitability
im = axes[2, 1].imshow(combined_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[2, 1].set_title('Combined Suitability (Weighted Overlay)', fontweight='bold')
plt.colorbar(im, ax=axes[2, 1], label='Score (0-100)')

# Hidden - remove
axes[2, 2].remove()

plt.suptitle('Suitability Analysis Layers', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'suitability_layers.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Suitability layers visualization saved')

In [ ]:
# Final classified map visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Continuous map
im1 = axes[0].imshow(combined_suitability, cmap='RdYlGn', vmin=0, vmax=100)
axes[0].set_title('Continuous Suitability Map (0-100)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('X Coordinate')
axes[0].set_ylabel('Y Coordinate')
plt.colorbar(im1, ax=axes[0], label='Suitability Score')

# Classified map
colors = [SUITABILITY_CLASSES[name][2] for name in SUITABILITY_CLASSES.keys()]
cmap = ListedColormap(colors)
im2 = axes[1].imshow(classified_suitability, cmap=cmap, vmin=0.5, vmax=5.5)
axes[1].set_title('Classified Suitability Map', fontsize=13, fontweight='bold')
axes[1].set_xlabel('X Coordinate')
axes[1].set_ylabel('Y Coordinate')

# Legend
patches = [mpatches.Patch(facecolor=SUITABILITY_CLASSES[name][2], 
                          label=name.replace('_', ' ').title())
          for name in SUITABILITY_CLASSES.keys()]
axes[1].legend(handles=patches, loc='upper right', fontsize=10)

plt.suptitle('School Site Suitability Maps - Chennai', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'final_suitability_maps.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Final suitability maps saved')

## 9. Statistical Analysis

In [ ]:
# Distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(combined_suitability.flatten(), bins=50, color='skyblue', edgecolor='black')
axes[0, 0].axvline(combined_suitability.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {combined_suitability.mean():.2f}')
axes[0, 0].set_xlabel('Suitability Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Suitability Scores')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Box plot by classification
data_for_box = [combined_suitability[classified_suitability == i+1] 
                 for i in range(len(SUITABILITY_CLASSES))]
axes[0, 1].boxplot(data_for_box, labels=[name.replace('_', '\n') for name in SUITABILITY_CLASSES.keys()])
axes[0, 1].set_ylabel('Suitability Score')
axes[0, 1].set_title('Score Distribution by Classification')
axes[0, 1].grid(True, alpha=0.3)

# Pie chart
class_counts = [np.sum(classified_suitability == i+1) for i in range(len(SUITABILITY_CLASSES))]
colors = [SUITABILITY_CLASSES[name][2] for name in SUITABILITY_CLASSES.keys()]
axes[1, 0].pie(class_counts, labels=SUITABILITY_CLASSES.keys(), autopct='%1.1f%%', colors=colors)
axes[1, 0].set_title('Area Distribution by Suitability Class')

# Cumulative distribution
sorted_scores = np.sort(combined_suitability.flatten())
cumulative = np.arange(1, len(sorted_scores) + 1) / len(sorted_scores) * 100
axes[1, 1].plot(sorted_scores, cumulative, linewidth=2, color='darkblue')
axes[1, 1].set_xlabel('Suitability Score')
axes[1, 1].set_ylabel('Cumulative Percentage')
axes[1, 1].set_title('Cumulative Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'statistical_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Statistical analysis saved')

## 10. Save Results to GeoTIFF

In [ ]:
def save_raster(data, filename, transform, crs):
    """Save numpy array as GeoTIFF"""
    output_path = OUTPUTS_DIR / filename
    
    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=crs,
        transform=transform
    ) as dst:
        dst.write(data, 1)
    
    print(f'✓ Saved: {filename}')
    return output_path

# Get CRS from original raster
with rasterio.open(PROCESSED_DATA_DIR / OUTPUT_FILES['lulc_raster']) as src:
    crs = src.crs
    transform = src.transform

# Save results
save_raster(combined_suitability, 'suitability_map.tif', transform, crs)
save_raster(classified_suitability, 'suitability_classified.tif', transform, crs)
save_raster(lulc_suitability, 'lulc_suitability.tif', transform, crs)
save_raster(pop_suitability, 'population_suitability.tif', transform, crs)
save_raster(slope_suitability, 'slope_suitability.tif', transform, crs)

print('\n✓ All results saved to outputs directory')

## 11. Summary and Recommendations

In [ ]:
print("\n" + "="*70)
print("SITE SUITABILITY ANALYSIS - SUMMARY REPORT")
print("="*70 + "\n")

print("SUITABILITY STATISTICS:")
print("-" * 70)
print(f"Minimum Score: {combined_suitability.min():.2f}")
print(f"Maximum Score: {combined_suitability.max():.2f}")
print(f"Mean Score: {combined_suitability.mean():.2f}")
print(f"Median Score: {np.median(combined_suitability):.2f}")
print(f"Standard Deviation: {combined_suitability.std():.2f}")

print("\nCLASSIFICATION BREAKDOWN:")
print("-" * 70)
for class_idx, (class_name, (min_val, max_val, color)) in enumerate(SUITABILITY_CLASSES.items(), 1):
    count = np.sum(classified_suitability == class_idx)
    percentage = (count / classified_suitability.size) * 100
    print(f"{class_name.replace('_', ' ').title():20} : {count:6} cells ({percentage:5.2f}%)")

print("\nTOP RECOMMENDATIONS:")
print("-" * 70)
print("1. Focus site selection on 'Highly Suitable' areas (85-100 score)")
print("2. Avoid 'Not Suitable' areas due to infrastructure constraints")
print("3. Consider field surveys to validate top candidate locations")
print("4. Conduct environmental impact assessment for selected sites")
print("5. Engage local stakeholders for final site approval")

print("\n" + "="*70)
print("Analysis Complete!")
print("="*70)